# WM-02 · IRIS 世界模型（修复版 · 真·梦境 GIF）

**上次失败原因（白话）**
1. 预训练权重 `Breakout.pt` **已经成功下载到 T4**。
2. 但代码用 `gymnasium.make('ALE/Breakout-...')` 开游戏时，**ALE 模拟器没注册好** → `Namespace ALE not found`。
3. 没有「真实第一帧」，官方 `WorldModelEnv` 就无法启动想象 → 只能导出空 GIF。

**本次修复**
- 用 `ale-py` + `gymnasium.register_envs` 正确打开 Breakout
- 取一帧真实画面 → `reset_from_initial_observations`
- 用 IRIS 的 tokenizer + Transformer 世界模型 **逐步预测下一帧**
- 导出可用 `iris_Breakout_dream.gif`

In [ ]:
import os, sys, subprocess, traceback
from pathlib import Path
import torch
print(sys.version)
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), '请打开 GPU'
print('GPU', torch.cuda.get_device_name(0))
WORK = Path('/kaggle/working')
OUT = WORK / 'wm_iris'
OUT.mkdir(exist_ok=True)
os.chdir(WORK)

def pip(*args):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *args]
    print('>>', ' '.join(args[:6]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        print((r.stderr or r.stdout or '')[-1500:])
    return r.returncode == 0

# 不碰系统 torch；装环境与 IRIS 依赖
pip('gymnasium', 'ale-py', 'autorom[accept-rom-license]', 'einops', 'opencv-python-headless', 'pillow', 'tqdm', 'huggingface-hub', 'imageio')
# ROM
subprocess.run([sys.executable, '-m', 'AutoROM', '--accept-license'], check=False)

# 验证 ALE
import gymnasium as gym
import ale_py
gym.register_envs(ale_py)
env = gym.make('ALE/Breakout-v5', render_mode='rgb_array')
obs, info = env.reset()
print('ALE OK', type(obs), getattr(obs, 'shape', None), 'actions', env.action_space.n)
env.close()

In [ ]:
import subprocess
from pathlib import Path
WORK = Path('/kaggle/working')
REPO = WORK / 'iris'
if not REPO.exists():
    subprocess.run(f'git clone --depth 1 https://github.com/eloialonso/iris.git {REPO}', shell=True, check=True)
print('repo', REPO, 'ok')

In [ ]:

import os, sys, traceback
from pathlib import Path
import numpy as np
import torch
from PIL import Image
import torchvision.transforms.functional as TF
import imageio.v2 as imageio

WORK = Path('/kaggle/working')
REPO = WORK / 'iris'
OUT = WORK / 'wm_iris'
OUT.mkdir(exist_ok=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'src'))

GAME = 'Breakout'
N_DREAM = 80
errors = []
ok = False

try:
    from huggingface_hub import hf_hub_download
    import gymnasium as gym
    import ale_py
    gym.register_envs(ale_py)

    # 彻底不 import hydra（Kaggle py3.12 + hydra1.1 会 dataclass 炸）
    _orig_load = torch.load
    def _load(*a, **k):
        k.setdefault('weights_only', False)
        try:
            return _orig_load(*a, **k)
        except TypeError:
            k.pop('weights_only', None)
            return _orig_load(*a, **k)
    torch.load = _load

    ckpt = Path(hf_hub_download('eloialonso/iris', f'pretrained_models/{GAME}.pt'))
    print('ckpt', ckpt, ckpt.stat().st_size)

    from models.tokenizer import Tokenizer, Encoder, Decoder, EncoderDecoderConfig
    from models.transformer import TransformerConfig
    from models.world_model import WorldModel
    from models.actor_critic import ActorCritic
    from agent import Agent
    from envs.world_model_env import WorldModelEnv

    enc_cfg = EncoderDecoderConfig(
        resolution=64,
        in_channels=3,
        z_channels=512,
        ch=64,
        ch_mult=[1, 1, 1, 1, 1],
        num_res_blocks=2,
        attn_resolutions=[8, 16],
        out_ch=3,
        dropout=0.0,
    )
    tokenizer = Tokenizer(
        vocab_size=512,
        embed_dim=512,
        encoder=Encoder(enc_cfg),
        decoder=Decoder(enc_cfg),
        with_lpips=False,  # 推理不需要 LPIPS
    )
    wm_cfg = TransformerConfig(
        tokens_per_block=17,
        max_blocks=20,
        attention='causal',
        num_layers=10,
        num_heads=4,
        embed_dim=256,
        embed_pdrop=0.1,
        resid_pdrop=0.1,
        attn_pdrop=0.1,
    )

    device = torch.device('cuda:0')
    raw = gym.make('ALE/Breakout-v5', render_mode='rgb_array', frameskip=4)
    obs0, _ = raw.reset()
    num_actions = int(raw.action_space.n)
    for _ in range(15):
        obs0, r, term, trunc, info = raw.step(raw.action_space.sample())
        if term or trunc:
            obs0, _ = raw.reset()
    raw.close()

    obs0_img = Image.fromarray(obs0).resize((64, 64), Image.BILINEAR)
    obs_init = TF.to_tensor(obs0_img).unsqueeze(0).to(device)
    print('init obs', tuple(obs_init.shape), 'num_actions', num_actions)

    world_model = WorldModel(
        obs_vocab_size=tokenizer.vocab_size,
        act_vocab_size=num_actions,
        config=wm_cfg,
    )
    actor_critic = ActorCritic(act_vocab_size=num_actions, use_original_obs=False)
    agent = Agent(tokenizer, world_model, actor_critic).to(device).eval()
    # 详细加载 + strict=False 兼容
    sd = torch.load(ckpt, map_location=device)
    from utils import extract_state_dict
    for name, module in [('tokenizer', agent.tokenizer), ('world_model', agent.world_model), ('actor_critic', agent.actor_critic)]:
        part = extract_state_dict(sd, name)
        missing, unexpected = module.load_state_dict(part, strict=False)
        print(name, 'keys', len(part), 'missing', len(missing), 'unexpected', len(unexpected))
        if missing[:5]:
            print('  missing sample', missing[:5])
        if unexpected[:5]:
            print('  unexpected sample', unexpected[:5])
    print('agent loaded OK (strict=False)')

    wm_env = WorldModelEnv(
        tokenizer=agent.tokenizer,
        world_model=agent.world_model,
        device=device,
        env=None,
    )
    obs = wm_env.reset_from_initial_observations(obs_init)

    def to_uint8(x):
        if torch.is_tensor(x):
            x = x.detach().float().cpu()
            if x.ndim == 4:
                x = x[0]
            if x.ndim == 3 and x.shape[0] in (1, 3):
                x = x.permute(1, 2, 0)
            x = x.numpy()
        x = np.asarray(x)
        if x.max() <= 1.5:
            x = x * 255.0
        return x.clip(0, 255).astype(np.uint8)

    frames = [np.array(obs0_img), to_uint8(obs)]
    rewards = []
    for t in range(N_DREAM):
        try:
            act = agent.act(obs if torch.is_tensor(obs) else obs_init, should_sample=True)
        except Exception as e:
            print('act fallback', e)
            act = torch.randint(0, num_actions, (1,), device=device)
        act = act.view(-1).to(device) if torch.is_tensor(act) else torch.tensor([int(act)], device=device)
        obs, rew, done, _ = wm_env.step(act)
        frames.append(to_uint8(obs))
        rewards.append(float(np.array(rew).reshape(-1)[0]))
        if (t + 1) % 20 == 0:
            print(f'dream step {t+1}/{N_DREAM} rew={rewards[-1]} done={bool(np.array(done).reshape(-1)[0])}')

    up = [np.array(Image.fromarray(f).resize((192, 192), Image.NEAREST)) for f in frames]
    gif = OUT / f'iris_{GAME}_dream.gif'
    imageio.mimsave(gif, up, fps=10, loop=0)
    idxs = np.linspace(0, len(up) - 1, num=min(10, len(up)), dtype=int)
    ims = [Image.fromarray(up[i]) for i in idxs]
    sheet = Image.new('RGB', (192 * len(ims), 192))
    for i, im in enumerate(ims):
        sheet.paste(im, (i * 192, 0))
    sheet_path = OUT / f'iris_{GAME}_strip.png'
    sheet.save(sheet_path)
    (OUT / 'summary.txt').write_text(
        f'game={GAME}\nckpt={ckpt}\nnum_actions={num_actions}\n'
        f'dream_steps={N_DREAM}\nframes={len(frames)}\n'
        f'reward_mean={float(np.mean(rewards)):.4f}\nreward_sum={float(np.sum(rewards)):.4f}\n'
        f'gif_bytes={gif.stat().st_size}\n',
        encoding='utf-8',
    )
    print('GIF', gif, 'frames', len(frames), 'bytes', gif.stat().st_size)
    print((OUT / 'summary.txt').read_text())
    try:
        from IPython.display import display, Image as IImage
        display(IImage(filename=str(gif)))
        display(sheet)
    except Exception:
        pass
    ok = gif.stat().st_size > 2000 and len(frames) > 10
except Exception:
    errors.append(traceback.format_exc())
    print(traceback.format_exc())

import shutil
shutil.make_archive(str(WORK / 'wm_iris_export'), 'zip', OUT)
print('ZIP', WORK / 'wm_iris_export.zip')
assert ok, 'IRIS dream failed:\n' + '\n'.join(errors)[:3500]
print('02 IRIS DREAM DONE')
